# 🦅 CIADA — Crow-Inspired Adaptive Displacement Algorithm

**Karga İlhamlı Adaptif Yer Değiştirme Algoritması**

---

| | |
|---|---|
| **Yazar** | Assist. Prof. Dr. Fuat CANDAN |
| **Kurum** | Istanbul Beykent University, Dept. of Computer Engineering |
| **ORCID** | 0000-0003-3166-0493 |

---

## İçindekiler
1. [Kurulum & İmport](#kurulum)
2. [Fitness Fonksiyonları](#fitness)
3. [CIADA Algoritması](#ciada)
4. [Tek Boyutlu Optimizasyon](#1d)
5. [Çok Boyutlu Bitki Besleme (N, W, pH)](#3d)
6. [10 Bitki Türü Karşılaştırması](#bitkiler)
7. [Mevsimsel Analiz — Domates Büyüme Evreleri](#mevsimsel)
8. [İstatistiksel Analiz — 30 Bağımsız Çalıştırma](#istatistik)
9. [Hassasiyet Analizi — α Katsayısı](#hassasiyet)
10. [Standart Benchmark Testleri (IEEE/CEC)](#benchmark)
11. [Yakınsama Grafikleri](#grafikler)
12. [Paralel Karmaşıklık Analizi](#karmasiklik)

## 1. Kurulum & İmport <a name='kurulum'></a>

In [ ]:
# Gerekli kütüphaneler (Colab'da zaten yüklü)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.gridspec import GridSpec
from dataclasses import dataclass
from typing import Callable, Optional, List
import time
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = 'white'
plt.rcParams['font.family']      = 'serif'

np.random.seed(42)
print('✅ Tüm kütüphaneler yüklendi.')

## 2. Fitness Fonksiyonları <a name='fitness'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BİTKİ BESLEME FITNESS FONKSİYONLARI
# ══════════════════════════════════════════════════════════════════

def gaussian_1d(ideal: float, sigma: float) -> Callable:
    """Tek boyutlu Gaussian fitness: f(x) = 100 × exp(−(x−ideal)² / 2σ²)"""
    def f(x):
        xv = float(np.atleast_1d(np.array(x, dtype=float))[0])
        return float(100.0 * np.exp(-((xv - ideal)**2) / (2*sigma**2)))
    return f

def plant_nutrition_fitness(ideal, sigma) -> Callable:
    """Çok boyutlu bağımsız ortalama: f(x) = mean(scoreᵢ)"""
    ideal_arr = np.array(ideal, dtype=float)
    sigma_arr = np.array(sigma, dtype=float)
    def f(x):
        x_arr  = np.array(x, dtype=float)
        scores = 100.0 * np.exp(-((x_arr - ideal_arr)**2) / (2*sigma_arr**2))
        return float(np.mean(scores))
    return f

# ══════════════════════════════════════════════════════════════════
# STANDART BENCHMARK FONKSİYONLARI (Maksimizasyon formatı)
# ══════════════════════════════════════════════════════════════════

def sphere(x):      return -float(np.sum(np.array(x)**2))
def rastrigin(x):
    x = np.array(x); d = len(x)
    return -(10*d + float(np.sum(x**2 - 10*np.cos(2*np.pi*x))))
def ackley(x):
    x = np.array(x); d = len(x)
    return -(- 20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
               - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e)
def rosenbrock(x):
    x = np.array(x)
    return -float(np.sum(100*(x[1:]-x[:-1]**2)**2 + (1-x[:-1])**2))
def schwefel(x):
    x = np.array(x); d = len(x)
    return -(418.9829*d - float(np.sum(x*np.sin(np.sqrt(np.abs(x))))))
def griewank(x):
    x = np.array(x)
    return -(np.sum(x**2)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1,len(x)+1)))) + 1)
def sine_1d(x):     return float(np.sin(np.atleast_1d(np.array(x,dtype=float))[0]))

# ══════════════════════════════════════════════════════════════════
# BİTKİ PROFİLLERİ (FAO 2022, USDA-NRCS 2023)
# ══════════════════════════════════════════════════════════════════
PLANT_PROFILES = {
    'domates':  {'ideal':[120,4.5,6.2], 'sigma':[25,1.5,0.5], 'lower':[0,0,4], 'upper':[250,15,9]},
    'misir':    {'ideal':[180,8.0,6.8], 'sigma':[25,2.0,0.6], 'lower':[0,0,4], 'upper':[300,20,9]},
    'marul':    {'ideal':[ 60,2.5,5.8], 'sigma':[15,0.8,0.4], 'lower':[0,0,4], 'upper':[150,10,9]},
    'bugday':   {'ideal':[ 95,4.0,6.5], 'sigma':[22,1.2,0.5], 'lower':[0,0,4], 'upper':[200,12,9]},
    'biber':    {'ideal':[ 75,3.2,6.0], 'sigma':[18,1.0,0.4], 'lower':[0,0,4], 'upper':[180,12,9]},
    'patates':  {'ideal':[110,5.5,6.1], 'sigma':[28,1.5,0.5], 'lower':[0,0,4], 'upper':[250,15,9]},
    'soya':     {'ideal':[ 65,3.8,6.3], 'sigma':[20,1.1,0.5], 'lower':[0,0,4], 'upper':[180,12,9]},
    'pamuk':    {'ideal':[130,6.2,6.3], 'sigma':[30,1.8,0.6], 'lower':[0,0,4], 'upper':[300,20,9]},
    'aycicegi': {'ideal':[ 88,4.1,6.4], 'sigma':[22,1.2,0.5], 'lower':[0,0,4], 'upper':[200,12,9]},
    'cilek':    {'ideal':[ 55,2.8,5.9], 'sigma':[16,0.9,0.4], 'lower':[0,0,4], 'upper':[150,10,9]},
}

TOMATO_STAGES = {
    'fide':       {'gun':'0-30',   'ideal':[45, 1.5,6.0], 'sigma':[12,0.5,0.3]},
    'vejetatif':  {'gun':'30-60',  'ideal':[75, 2.8,6.1], 'sigma':[18,0.8,0.4]},
    'ciceklenme': {'gun':'60-90',  'ideal':[95, 3.5,6.2], 'sigma':[20,1.0,0.4]},
    'meyve':      {'gun':'90-120', 'ideal':[120,4.5,6.2], 'sigma':[25,1.5,0.5]},
    'hasat':      {'gun':'120-150','ideal':[85, 3.0,6.3], 'sigma':[20,1.0,0.4]},
}

print('✅ Fitness fonksiyonları ve bitki profilleri yüklendi.')
print(f'   Bitki türleri  : {list(PLANT_PROFILES.keys())}')
print(f'   Domates evreleri: {list(TOMATO_STAGES.keys())}')

## 3. CIADA Algoritması <a name='ciada'></a>

In [ ]:
class CIADAOptimizer:
    """
    CIADA — Crow-Inspired Adaptive Displacement Algorithm

    Formül: ΔV = (U − L) × e^(−2t/G) × Rand(0,1)   [α = 2.0]
    Genel : ΔV = (U − L) × e^(−αt/G) × Rand(0,1)

    Parametreler
    ------------
    n     : Popülasyon büyüklüğü (önerilen: 20)
    G     : Maks. iterasyon      (önerilen: 50)
    alpha : Sönümleme katsayısı  (güvenli aralık: [1.5, 3.0], önerilen: 2.0)
    seed  : Rastgele tohum (None = her seferinde farklı)
    """
    def __init__(self, n=20, G=50, alpha=2.0, seed=None):
        self.n, self.G, self.alpha, self.seed = n, G, alpha, seed

    def run(self, lower, upper, fitness_fn, target=99.0):
        if self.seed is not None:
            np.random.seed(self.seed)

        L = np.atleast_1d(np.array(lower, dtype=float))
        U = np.atleast_1d(np.array(upper, dtype=float))
        d = len(L)

        # P(t=0) ~ Uniform(L, U)
        pop       = np.random.uniform(L, U, (self.n, d))
        best_fit  = -np.inf
        best_x    = pop[0].copy()
        history   = []
        conv_iter = -1

        for t in range(self.G):
            fits = np.array([fitness_fn(pop[i]) for i in range(self.n)])
            idx  = np.argmax(fits)
            if fits[idx] > best_fit:
                best_fit = fits[idx]
                best_x   = pop[idx].copy()
            history.append(best_fit)

            if best_fit >= target and conv_iter == -1:
                conv_iter = t
                break

            # ΔV = (U−L) × e^(−αt/G) × Rand(0,1)   [α=2.0]
            dv = (U - L) * np.exp(-self.alpha * t / self.G) * np.random.rand(d)

            new_pop = np.empty_like(pop)
            for i in range(self.n):
                if fits[i] < best_fit:
                    # Stratejik Atış (Exploitation)
                    new_pop[i] = pop[i] + dv * (best_x - pop[i]) * np.random.rand(d)
                else:
                    # Keşif Modu (Exploration)
                    new_pop[i] = pop[i] + np.random.uniform(-1,1,d) * dv
                # Sınır Kontrolü — Kap Çeperi
                new_pop[i] = np.clip(new_pop[i], L, U)
            pop = new_pop

        return {'best_x': best_x, 'best_fit': best_fit,
                'conv_iter': conv_iter, 'history': history}

    def run_multiple(self, n_runs=30, **kwargs):
        """n_runs kez bağımsız çalıştır, istatistik döndür."""
        results = []
        for s in range(n_runs):
            opt = CIADAOptimizer(self.n, self.G, self.alpha, seed=s)
            results.append(opt.run(**kwargs))
        fits = np.array([r['best_fit'] for r in results])
        return {'results': results, 'fits': fits,
                'mean': fits.mean(), 'std': fits.std(),
                'min': fits.min(), 'max': fits.max(),
                'median': np.median(fits)}

print('✅ CIADAOptimizer sınıfı yüklendi.')
print('   Kullanım: opt = CIADAOptimizer(n=20, G=50, alpha=2.0, seed=42)')
print('             r   = opt.run(lower=0, upper=150, fitness_fn=fn, target=99.0)')

## 4. Tek Boyutlu Optimizasyon <a name='1d'></a>

Azot optimizasyonu: hedef 85 mg/kg, aralık [0, 150]

In [ ]:
fn1   = gaussian_1d(ideal=85.0, sigma=25.0)
opt   = CIADAOptimizer(n=20, G=50, alpha=2.0, seed=42)
r1    = opt.run(lower=0, upper=150, fitness_fn=fn1, target=99.0)

print('─'*45)
print('  Tek Boyutlu Azot Optimizasyonu')
print('─'*45)
print(f'  Optimum x    : {r1["best_x"][0]:.4f} mg/kg')
print(f'  Fitness      : {r1["best_fit"]:.4f}%')
ci = r1['conv_iter'] if r1['conv_iter']>=0 else 'ulaşılamadı'
print(f'  Yakınsama    : {ci}. iterasyon')
print(f'  Hata (|x̂-x*)  : {abs(r1["best_x"][0]-85):.4f} mg/kg')

# Grafik
fig, axes = plt.subplots(1, 2, figsize=(13,5))
x_vals = np.linspace(0, 150, 400)
axes[0].plot(x_vals, [fn1(x) for x in x_vals], color='#1F3864', lw=2, label='Fitness eğrisi')
axes[0].axvline(85, color='red', ls='--', lw=1.5, label='Hedef (85 mg/kg)')
axes[0].axvline(r1['best_x'][0], color='green', ls=':', lw=2, label=f'CIADA: {r1["best_x"][0]:.2f}')
axes[0].set_xlabel('Azot (mg/kg)'); axes[0].set_ylabel('Fitness (%)')
axes[0].set_title('Fitness Yüzeyi ve CIADA Sonucu'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, len(r1['history'])+1), r1['history'], color='#1F3864', lw=2.5)
axes[1].axhline(99, color='gray', ls='--', lw=1, label='%99 Eşiği')
axes[1].set_xlabel('İterasyon'); axes[1].set_ylabel('Fitness (%)')
axes[1].set_title('Yakınsama Eğrisi'); axes[1].legend(); axes[1].grid(alpha=0.3)
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 5. Çok Boyutlu Bitki Besleme (N, W, pH) <a name='3d'></a>

In [ ]:
profile = PLANT_PROFILES['domates']
fn3d    = plant_nutrition_fitness(profile['ideal'], profile['sigma'])
opt3d   = CIADAOptimizer(n=20, G=50, alpha=2.0, seed=42)
r3d     = opt3d.run(lower=profile['lower'], upper=profile['upper'],
                    fitness_fn=fn3d, target=99.0)

ideal = profile['ideal']
print('─'*50)
print('  Çok Boyutlu Domates Optimizasyonu (N, W, pH)')
print('─'*50)
labels = ['N (mg/kg)', 'W (L/gün)', 'pH']
for i, lbl in enumerate(labels):
    found = r3d['best_x'][i]
    rmse  = abs(found - ideal[i])
    print(f'  {lbl:12s}: Bulunan={found:.4f}  İdeal={ideal[i]}  |e|={rmse:.4f}')
print(f'  Fitness      : {r3d["best_fit"]:.4f}%')

# 30 çalıştırma RMSE
stats3d = opt3d.run_multiple(30, lower=profile['lower'], upper=profile['upper'],
                              fitness_fn=fn3d, target=99.0)
all_xs  = np.array([r['best_x'] for r in stats3d['results']])
print('\n  30 Çalıştırma RMSE:')
for i, lbl in enumerate(labels):
    rmse = np.sqrt(np.mean((all_xs[:,i] - ideal[i])**2))
    print(f'  {lbl:12s}: RMSE = {rmse:.4f}')
print(f'  Ort. Fitness : {stats3d["mean"]:.4f}% ± {stats3d["std"]:.4f}')

## 6. 10 Bitki Türü Karşılaştırması <a name='bitkiler'></a>

In [ ]:
results_plants = {}
print(f'  {"Bitki":12s}  {"Opt.N":>8s}  {"Opt.W":>8s}  {"pH":>6s}  {"Fitness":>10s}')
print('  ' + '─'*52)
for name, prof in PLANT_PROFILES.items():
    fn  = plant_nutrition_fitness(prof['ideal'], prof['sigma'])
    opt = CIADAOptimizer(n=20, G=50, alpha=2.0, seed=42)
    r   = opt.run(lower=prof['lower'], upper=prof['upper'], fitness_fn=fn)
    results_plants[name] = r
    bx = r['best_x']
    print(f'  {name:12s}  {bx[0]:8.2f}  {bx[1]:8.2f}  {bx[2]:6.2f}  {r["best_fit"]:9.2f}%')

# Grafik
fig, ax = plt.subplots(figsize=(13,5))
colors = plt.cm.tab10(np.linspace(0,1,10))
for i, (name, r) in enumerate(results_plants.items()):
    ax.plot(range(1, len(r['history'])+1), r['history'],
            color=colors[i], lw=2, label=name)
ax.axhline(99, color='gray', ls='--', lw=1, alpha=0.7, label='%99 eşiği')
ax.set_xlabel('İterasyon', fontsize=12); ax.set_ylabel('Fitness (%)', fontsize=12)
ax.set_title('CIADA — 10 Bitki Türünde Yakınsama Eğrileri', fontsize=13, color='#1F3864')
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Mevsimsel Analiz — Domates Büyüme Evreleri <a name='mevsimsel'></a>

In [ ]:
L_t = [0, 0, 4.0]; U_t = [250, 15, 9.0]
results_stages = {}
print(f'  {"Evre":15s}  {"Gün":>10s}  {"N(mg/kg)":>10s}  {"W(L/gün)":>9s}  {"pH":>6s}  {"Fitness":>9s}')
print('  ' + '─'*68)
for stage, cfg in TOMATO_STAGES.items():
    fn  = plant_nutrition_fitness(cfg['ideal'], cfg['sigma'])
    opt = CIADAOptimizer(n=20, G=50, alpha=2.0, seed=42)
    r   = opt.run(lower=L_t, upper=U_t, fitness_fn=fn)
    results_stages[stage] = r
    bx = r['best_x']
    print(f'  {stage:15s}  {cfg["gun"]:>10s}  {bx[0]:10.2f}  {bx[1]:9.2f}  {bx[2]:6.2f}  {r["best_fit"]:8.2f}%')

# Grafik
fig, axes = plt.subplots(1, 2, figsize=(14,5))
stage_colors = ['#8E44AD','#2980B9','#27AE60','#E67E22','#C0392B']
for i, (stage, r) in enumerate(results_stages.items()):
    axes[0].plot(range(1, len(r['history'])+1), r['history'],
                 color=stage_colors[i], lw=2.2, label=stage)
axes[0].axhline(99, color='gray', ls='--', lw=1, alpha=0.6)
axes[0].set_xlabel('İterasyon'); axes[0].set_ylabel('Fitness (%)')
axes[0].set_title('Büyüme Evrelerine Göre Yakınsama'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

n_ideals = [TOMATO_STAGES[s]['ideal'][0] for s in TOMATO_STAGES]
axes[1].bar(list(TOMATO_STAGES.keys()), n_ideals, color=stage_colors, edgecolor='white', lw=1.5)
for i, v in enumerate(n_ideals):
    axes[1].text(i, v+1.5, f'{v}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Optimum Azot (mg/kg)'); axes[1].set_xlabel('Büyüme Evresi')
axes[1].set_title('Evreye Göre Optimum N Gereksinimi'); axes[1].grid(alpha=0.3, axis='y')
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 8. İstatistiksel Analiz — 30 Bağımsız Çalıştırma <a name='istatistik'></a>

In [ ]:
fn_stat = gaussian_1d(ideal=85.0, sigma=25.0)
kw = dict(lower=0, upper=150, fitness_fn=fn_stat, target=99.0)

def run_algo_pso(n=20, G=50, seed=0):
    np.random.seed(seed)
    pos = np.random.uniform(0,150,n); vel = np.zeros(n)
    pb  = pos.copy(); pbf = np.array([fn_stat(x) for x in pos])
    gb  = pos[np.argmax(pbf)]; gbf = pbf.max(); hist=[]
    for t in range(G):
        w = 0.9-(0.9-0.4)*t/G
        fits = np.array([fn_stat(x) for x in pos])
        for i in range(n):
            if fits[i]>pbf[i]: pb[i]=pos[i]; pbf[i]=fits[i]
            if fits[i]>gbf:    gb=pos[i];    gbf=fits[i]
        hist.append(gbf)
        vel = w*vel+2*np.random.rand(n)*(pb-pos)+2*np.random.rand(n)*(gb-pos)
        pos = np.clip(pos+vel, 0, 150)
    return gbf

def run_algo_ga(n=20, G=50, seed=0):
    np.random.seed(seed)
    pop = np.random.uniform(0,150,n); best=-np.inf; hist=[]
    for _ in range(G):
        fits = np.array([fn_stat(x) for x in pop])
        best = max(best, fits.max()); hist.append(best)
        new=[]
        for _ in range(n):
            p1,p2=pop[np.random.randint(n)],pop[np.random.randint(n)]
            new.append(np.clip(0.5*(p1+p2)+np.random.normal(0,5),0,150))
        pop=np.array(new)
    return best

print('30 bağımsız çalıştırma yapılıyor...', end=' ')
stats_ciada = CIADAOptimizer(20,50,2.0).run_multiple(30, **kw)
fits_pso  = np.array([run_algo_pso(seed=s) for s in range(30)])
fits_ga   = np.array([run_algo_ga(seed=s)  for s in range(30)])
print('Tamamlandı!\n')

print(f'  {"Algoritma":8s}  {"Ortalama":>10s}  {"Std":>8s}  {"Min":>8s}  {"Max":>8s}')
print('  ' + '─'*50)
for algo, fits in [("CIADA", stats_ciada['fits']), ("PSO", fits_pso), ("GA", fits_ga)]:
    print(f'  {algo:8s}  {fits.mean():10.4f}  {fits.std():8.4f}  {fits.min():8.4f}  {fits.max():8.4f}')

# Box plot
fig, axes = plt.subplots(1,2, figsize=(13,5))
bp = axes[0].boxplot([stats_ciada['fits'], fits_pso, fits_ga],
                     labels=['CIADA','PSO','GA'], patch_artist=True, notch=True,
                     medianprops=dict(color='white',lw=2.5))
for patch, c in zip(bp['boxes'], ['#1F3864','#27AE60','#C0392B']):
    patch.set_facecolor(c); patch.set_alpha(0.75)
axes[0].set_ylabel('Fitness (%)'); axes[0].set_title('Final Fitness Dağılımı (n=30)')
axes[0].grid(alpha=0.3,axis='y'); axes[0].spines[['top','right']].set_visible(False)

# Yakınsama karşılaştırması (ilk run)
np.random.seed(0)
r_c = CIADAOptimizer(20,50,2.0,seed=0).run(**kw)
hist_pso=[]; pos=np.random.uniform(0,150,20); vel=np.zeros(20)
pb=pos.copy(); pbf=np.array([fn_stat(x) for x in pos]); gb=pos[pbf.argmax()]; gbf=pbf.max()
for t in range(50):
    w=0.9-(0.9-0.4)*t/50; fits=np.array([fn_stat(x) for x in pos])
    for i in range(20):
        if fits[i]>pbf[i]: pb[i]=pos[i]; pbf[i]=fits[i]
        if fits[i]>gbf:    gb=pos[i];    gbf=fits[i]
    hist_pso.append(gbf)
    vel=w*vel+2*np.random.rand(20)*(pb-pos)+2*np.random.rand(20)*(gb-pos)
    pos=np.clip(pos+vel,0,150)

for hist, label, color in [
    (r_c['history'],'CIADA','#1F3864'),
    (hist_pso,'PSO','#27AE60'),
]:
    axes[1].plot(range(1,len(hist)+1), hist, color=color, lw=2, label=label)
axes[1].axhline(99, color='gray', ls='--', lw=1, label='%99 eşiği')
axes[1].set_xlabel('İterasyon'); axes[1].set_ylabel('Fitness (%)')
axes[1].set_title('Yakınsama Karşılaştırması'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 9. Hassasiyet Analizi — α Katsayısı <a name='hassasiyet'></a>

In [ ]:
fn_s = gaussian_1d(ideal=85.0, sigma=25.0)
alphas = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

print(f'  {"α":>5s}  {"Ort. Fitness":>13s}  {"Std":>8s}  {"Yakınsama":>10s}  {"Yorum"}')
print('  ' + '─'*65)
alpha_results = []
for a in alphas:
    fits = [CIADAOptimizer(20,50,a,seed=s).run(0,150,fn_s)['best_fit'] for s in range(30)]
    fits = np.array(fits)
    conv = np.mean([CIADAOptimizer(20,50,a,seed=s).run(0,150,fn_s,99)['conv_iter']
                    for s in range(30)])
    note = '← Önerilen' if a==2.0 else ('Yetersiz' if a<1.5 else ('Erken sönümleme' if a>3.0 else 'İyi'))
    alpha_results.append(fits.mean())
    print(f'  {a:>5.1f}  {fits.mean():13.4f}  {fits.std():8.4f}  {conv:10.1f}  {note}')

fig, ax = plt.subplots(figsize=(10,5))
bar_colors = ['#C0392B' if (a<1.5 or a>3.0) else ('#1F3864' if a==2.0 else '#5B8DB8') for a in alphas]
ax.bar([str(a) for a in alphas], alpha_results, color=bar_colors, edgecolor='white', lw=1.5, zorder=3)
ax.axvspan(1.5, 4.5, alpha=0.07, color='green', label='Güvenli aralık [1.5–3.0]')
ax.axhline(99, color='gray', ls='--', lw=1, label='%99 eşiği')
for i, v in enumerate(alpha_results):
    ax.text(i, v+0.1, f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xlabel('α (Sönümleme Katsayısı)', fontsize=12); ax.set_ylabel('Ort. Fitness (%)', fontsize=12)
ax.set_title('α Hassasiyet Analizi — Güvenli Çalışma Bölgesi', fontsize=13, color='#1F3864')
ax.set_ylim(88, 102); ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.spines[['top','right']].set_visible(False); plt.tight_layout(); plt.show()

## 10. Standart Benchmark Testleri (IEEE/CEC) <a name='benchmark'></a>

In [ ]:
BENCHMARKS = {
    'sphere'    : (sphere,     -5.12,  5.12,  0.0,     'Unimodal'),
    'rastrigin' : (rastrigin,  -5.12,  5.12,  0.0,     'Multimodal'),
    'schwefel'  : (schwefel,   -500,   500,   420.968, 'Deceptive'),
    'sine'      : (sine_1d,    0,      6.28,  1.5708,  'Unimodal'),
    'ackley'    : (ackley,     -32,    32,    0.0,     'Multimodal'),
    'rosenbrock': (rosenbrock, -2,     2,     1.0,     'Valley'),
    'griewank'  : (griewank,   -600,   600,   0.0,     'Periodic'),
}

print(f'  {"Fonksiyon":12s}  {"Sınıf":12s}  {"x* Hatası":>10s}  {"Ort. Fitness":>13s}  {"Değerlendirme"}')
print('  ' + '─'*72)
bm_results = {}
for name, (fn, L, U, x_opt, cls) in BENCHMARKS.items():
    is_1d = name in ('sine', 'schwefel')
    kw_b  = dict(lower=L, upper=U, fitness_fn=fn, target=0)
    if not is_1d:
        kw_b = dict(lower=[L,L], upper=[U,U], fitness_fn=fn, target=0)
    stats = CIADAOptimizer(30,100,2.0).run_multiple(10, **kw_b)
    best  = stats['results'][np.argmax(stats['fits'])]
    bx    = best['best_x']
    err   = abs(bx[0] - x_opt)
    rating = 'Mükemmel ✓' if err < 0.1 else ('İyi ✓' if err < 1.0 else 'Sınırlı ✗')
    bm_results[name] = stats
    print(f'  {name:12s}  {cls:12s}  {err:10.4f}  {stats["mean"]:13.4f}  {rating}')

## 11. Yakınsama Grafikleri — Tüm Benchmark <a name='grafikler'></a>

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
colors_bm = plt.cm.Set1(np.linspace(0,1,7))

for idx, (name, (fn, L, U, x_opt, cls)) in enumerate(BENCHMARKS.items()):
    ax = axes[idx]
    is_1d = name in ('sine', 'schwefel')
    kw_b = dict(lower=L, upper=U, fitness_fn=fn)
    if not is_1d:
        kw_b = dict(lower=[L,L], upper=[U,U], fitness_fn=fn)

    # 5 çalıştırma ortalaması
    all_h = []
    for s in range(5):
        r = CIADAOptimizer(30,100,2.0,seed=s).run(**kw_b)
        all_h.append(r['history'])
    max_len = max(len(h) for h in all_h)
    padded  = [h + [h[-1]]*(max_len-len(h)) for h in all_h]
    mean_h  = np.mean(padded, axis=0)

    ax.plot(range(1,len(mean_h)+1), mean_h, color=colors_bm[idx], lw=2)
    ax.fill_between(range(1,len(mean_h)+1),
                    np.min(padded,0), np.max(padded,0), alpha=0.1, color=colors_bm[idx])
    ax.set_title(f'{name.capitalize()} ({cls})', fontsize=10, fontweight='bold', color='#1F3864')
    ax.set_xlabel('İterasyon', fontsize=9); ax.set_ylabel('Fitness', fontsize=9)
    ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)

axes[-1].axis('off')
plt.suptitle('CIADA — IEEE/CEC Benchmark Yakınsama Eğrileri (n=5 çalıştırma ort.)',
             fontsize=13, fontweight='bold', color='#1F3864')
plt.tight_layout(); plt.show()

## 12. Paralel Karmaşıklık Analizi <a name='karmasiklik'></a>

In [ ]:
# n ve G'ye göre gerçek çalışma süresi
fn_t = gaussian_1d(85.0, 25.0)
n_vals = [5,10,20,30,50,100,200]
G_vals = [10,20,50,75,100]

times_n, times_G = [], []
for n in n_vals:
    runs=[]
    for _ in range(5):
        t0=time.perf_counter()
        CIADAOptimizer(n,50,2.0,seed=0).run(0,150,fn_t)
        runs.append((time.perf_counter()-t0)*1000)
    times_n.append(np.mean(runs))

for G in G_vals:
    runs=[]
    for _ in range(5):
        t0=time.perf_counter()
        CIADAOptimizer(20,G,2.0,seed=0).run(0,150,fn_t)
        runs.append((time.perf_counter()-t0)*1000)
    times_G.append(np.mean(runs))

print('  n vs Süre (G=50 sabit):')
for n,t in zip(n_vals,times_n):
    print(f'    n={n:4d}: {t:.3f} ms')
print('\n  G vs Süre (n=20 sabit):')
for G,t in zip(G_vals,times_G):
    print(f'    G={G:4d}: {t:.3f} ms')

fig, axes = plt.subplots(1,2, figsize=(13,5))
for ax, vals, times, xlabel, title in [
    (axes[0], n_vals, times_n, 'Popülasyon Büyüklüğü (n)', 'n vs Süre (G=50 sabit)'),
    (axes[1], G_vals, times_G, 'Maksimum İterasyon (G)',   'G vs Süre (n=20 sabit)'),
]:
    ax.plot(vals, times, 'o-', color='#1F3864', lw=2.5, ms=7, label='Ölçülen')
    theory = [times[0]/vals[0]*v for v in vals]
    ax.plot(vals, theory, '--', color='#C0392B', lw=1.5, label='Teorik O(n)')
    ax.set_xlabel(xlabel, fontsize=11); ax.set_ylabel('Süre (ms)', fontsize=11)
    ax.set_title(title, fontsize=12, color='#1F3864')
    ax.legend(); ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)
    for x,t in zip(vals,times):
        ax.annotate(f'{t:.1f}ms',(x,t),textcoords='offset points',
                    xytext=(0,8),ha='center',fontsize=8)
plt.tight_layout(); plt.show()

print('\n✅ Tüm analizler tamamlandı!')